In [1]:
import requests
import pandas as pd
import re
import time
from bs4 import BeautifulSoup
import io
from tqdm import tqdm

In [2]:
url = "https://www.basketball-reference.com/teams/"
headers= {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers, timeout=20)
print(response.status_code)
soup = BeautifulSoup(response.text, "html.parser")

200


In [3]:
team_ids = ['ATL','BOS','NJN','CHA','CHI','CLE','DAL','DEN','DET','GSW','HOU','IND','LAC','LAL','MEM',
 'MIA','MIL','MIN','NOH','NYK','OKC','ORL','PHI','PHO','POR','SAC','SAS','TOR','UTA','WAS']

In [4]:
team_links = []

for team_id in team_ids:

    link = ("https://www.basketball-reference.com/teams/" + team_id + "/")
    team_links.append(link)

team_links

['https://www.basketball-reference.com/teams/ATL/',
 'https://www.basketball-reference.com/teams/BOS/',
 'https://www.basketball-reference.com/teams/NJN/',
 'https://www.basketball-reference.com/teams/CHA/',
 'https://www.basketball-reference.com/teams/CHI/',
 'https://www.basketball-reference.com/teams/CLE/',
 'https://www.basketball-reference.com/teams/DAL/',
 'https://www.basketball-reference.com/teams/DEN/',
 'https://www.basketball-reference.com/teams/DET/',
 'https://www.basketball-reference.com/teams/GSW/',
 'https://www.basketball-reference.com/teams/HOU/',
 'https://www.basketball-reference.com/teams/IND/',
 'https://www.basketball-reference.com/teams/LAC/',
 'https://www.basketball-reference.com/teams/LAL/',
 'https://www.basketball-reference.com/teams/MEM/',
 'https://www.basketball-reference.com/teams/MIA/',
 'https://www.basketball-reference.com/teams/MIL/',
 'https://www.basketball-reference.com/teams/MIN/',
 'https://www.basketball-reference.com/teams/NOH/',
 'https://ww

In [5]:
def get_team_seasons(team_link, team_id):

    response = requests.get(team_link, headers=headers, timeout=20)

    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    table = soup.find("table", id=team_id)

    if table is None:
        print("Table not found:", team_id)
        return []

    data = []

    rows = table.find("tbody").find_all("tr")

  
    rows = rows[:7]

    for row in rows:

        season_cell = row.find("th")

        if season_cell is None:
            continue

        season = season_cell.get_text(strip=True)

        cells = row.find_all("td")

        values = []

        for cell in cells:
            text = cell.get_text(strip=True)
            values.append(text)

        row_data = {
            "team_id": team_id,
            "season": season
        }

        for cell in cells:

            column_name = cell.get("data-stat")
            value = cell.get_text(strip=True)

            if column_name is not None:
                row_data[column_name] = value

        data.append(row_data)

    return data

In [6]:
all_team_seasons = []

for i in tqdm(range(len(team_links))):

    team_link = team_links[i]
    team_id = team_ids[i]

    team_data = get_team_seasons(
        team_link,
        team_id
    )

    if len(team_data) > 0:
        all_team_seasons.extend(team_data)

    time.sleep(5)

  0%|          | 0/30 [00:00<?, ?it/s]

100%|██████████| 30/30 [02:56<00:00,  5.87s/it]


In [7]:
team_seasons = pd.DataFrame(all_team_seasons)

team_seasons.head()

,team_id,season,lg_id,team_name,wins,losses,win_loss_pct,rank_team,srs,DUMMY,pace,pace_rel,off_rtg,off_rtg_rel,def_rtg,def_rtg_rel,rank_team_playoffs,coaches,top_ws
0,ATL,2025-26,NBA,Atlanta Hawks*,46,36,.561,1st of 5,2.38,,101.7,2.3,116.1,0.3,113.7,-2.1,Lost E. Conf. 1st Rnd.,Q. Snyder(46-36),J. Johnson(7.5)
1,ATL,2024-25,NBA,Atlanta Hawks,40,42,.488,2nd of 5,-1.41,,102.6,3.8,114.6,0.1,115.7,1.2,,Q. Snyder(40-42),O. Okongwu(7.2)
2,ATL,2023-24,NBA,Atlanta Hawks,36,46,.439,3rd of 5,-2.38,,100.1,1.6,117.2,1.9,119.4,4.1,,Q. Snyder(36-46),C. Capela(6.3)
3,ATL,2022-23,NBA,Atlanta Hawks*,41,41,.500,2nd of 5,0.32,,100.7,1.6,116.6,1.8,116.3,1.5,Lost E. Conf. 1st Rnd.,"N. McMillan(29-30),J. Prunty(2-0),Q. Snyder(10...",C. Capela(7.2)
4,ATL,2021-22,NBA,Atlanta Hawks*,43,39,.524,2nd of 5,1.55,,97.7,-0.5,116.5,4.5,114.9,2.9,Lost E. Conf. 1st Rnd.,N. McMillan(43-39),T. Young(10.0)


In [8]:
team_seasons.to_csv("team_seasons.csv", index=False, encoding="utf-8")